# Temporal CNN: projection = channels
Thử P=C=256 và P=C=512. Giữ kernel 5, scoring head, residual on, Top-K và loss của lượt trước. Train từ scratch tối đa 30 epoch; seed mô hình và DataLoader đều 234. Early stopping chỉ dùng train loss để nhận biết hội tụ, không dùng test AUC và không phát hiện overfitting. Kết quả P=C=128 cũ chỉ là tham khảo vì lịch train khác.

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path
import torch
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Finetune VadCLIP')
SRC_DIR = PROJECT_ROOT / 'VadCLIP' / 'src'
LIST_DIR = PROJECT_ROOT / 'VadCLIP' / 'list'
FEATURE_ROOT = Path('/content/UCFClipFeatures')
OUTPUT_ROOT = PROJECT_ROOT / 'code' / 'results'
RUN_TAG = 'cnn_k5_equal_width_long_s234'
WIDTHS = (256, 512)
SEED = 234
DATA_SEED = 234
MAX_EPOCH = 30
MIN_EPOCHS = 20
PATIENCE = 6
MIN_DELTA = 0.001
MILESTONES = [10, 20]

if not torch.cuda.is_available():
    raise RuntimeError('Hãy chọn GPU runtime trên Colab.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'scipy', 'scikit-learn', 'pandas'], check=True)
if not FEATURE_ROOT.is_dir():
    archives = [PROJECT_ROOT / name for name in ('UCFClipFeatures.zip', 'UCFClipFeatures.tar', 'UCFClipFeatures.tar.gz')]
    archive = next((path for path in archives if path.is_file()), None)
    if archive is None:
        raise FileNotFoundError('Không tìm thấy UCFClipFeatures hoặc file nén trên Drive.')
    local_archive = Path('/content') / archive.name
    if not local_archive.is_file() or local_archive.stat().st_size != archive.stat().st_size:
        shutil.copy2(archive, local_archive)
    command = ['unzip', '-q', str(local_archive), '-d', '/content'] if archive.suffix == '.zip' else ['tar', '-xf', str(local_archive), '-C', '/content']
    subprocess.run(command, check=True)
required = (SRC_DIR / 'model_cnn_vadclip.py', SRC_DIR / 'ucf_train_cnn_vadclip.py',
            SRC_DIR / 'ucf_evaluate_cnn_vadclip.py', SRC_DIR / 'ucf_cnn_common.py',
            LIST_DIR / 'ucf_CLIP_rgb_relative.csv', LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv',
            LIST_DIR / 'gt_ucf.npy', LIST_DIR / 'gt_segment_ucf.npy', LIST_DIR / 'gt_label_ucf.npy', FEATURE_ROOT)
for path in required:
    if not path.exists():
        raise FileNotFoundError(path)
print('GPU:', torch.cuda.get_device_name(0))
print('Feature root:', FEATURE_ROOT)
print('Run tag:', RUN_TAG)


## Smoke test
Kiểm tra shape [C,C,5], residual không chiếu thêm và gradient hữu hạn.

In [ ]:
sys.path.insert(0, str(SRC_DIR))
from model_cnn_vadclip import TemporalCNNClassifier
for width in WIDTHS:
    cnn = TemporalCNNClassifier(projection_width=width, hidden_width=width, kernel_size=5, use_residual=True)
    assert tuple(cnn.projection[0].weight.shape) == (width, 512)
    assert tuple(cnn.blocks[0].conv1.weight.shape) == (width, width, 5)
    assert tuple(cnn.blocks[1].conv2.weight.shape) == (width, width, 5)
    assert isinstance(cnn.blocks[0].skip_projection, torch.nn.Identity)
    scores = cnn(torch.randn(2, 12, 512), torch.tensor([8, 12]))
    assert scores.shape == (2, 12, 1) and torch.isfinite(scores).all()
    assert torch.equal(scores[0, 8:], torch.zeros_like(scores[0, 8:]))
    scores[:, :8].sum().backward()
    assert cnn.blocks[0].conv1.weight.grad is not None
    print('Smoke test PASS: P=C=', width)
del cnn


## Train và evaluate
Mỗi cấu hình lưu checkpoint từng epoch và model có train loss tốt nhất vào thư mục riêng. Evaluate đúng model này một lần sau train; không dùng test để dừng sớm. Nếu train loss vẫn tiếp tục giảm, chạy đủ 30 epoch.

In [ ]:
def run_command(command):
    print('Running:', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=SRC_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait() != 0:
        raise RuntimeError('Tiến trình thất bại; traceback gốc đã in ở trên.')

results = {}
for width in WIDTHS:
    variant = f'k5_head_topk_p{width}_c{width}'
    checkpoint_dir = SRC_DIR / 'model' / 'epoch_checkpoints_cnn_vadclip' / RUN_TAG / variant
    output_model = SRC_DIR / 'model' / 'cnn_vadclip' / RUN_TAG / f'{variant}.pth'
    train_dir = OUTPUT_ROOT / 'train' / RUN_TAG / variant
    eval_dir = OUTPUT_ROOT / 'evaluate' / RUN_TAG / variant
    analyze_dir = OUTPUT_ROOT / 'analyze' / RUN_TAG / variant
    config_path = train_dir / 'config.json'
    summary_path = train_dir / 'early_stopping.json'
    expected = {'projection_width': width, 'cnn_channels': width, 'kernel_size': 5,
                'score_mode': 'scoring_head', 'residual_mode': 'on', 'aggregation': 'topk',
                'lambda_alignment': 1.0, 'lambda_text': 1.0, 'seed': SEED,
                'data_seed': DATA_SEED, 'batch_size': 64, 'lr': 1e-5,
                'max_epoch': MAX_EPOCH, 'min_epochs': MIN_EPOCHS,
                'early_stopping_patience': PATIENCE, 'early_stopping_min_delta': MIN_DELTA,
                'scheduler_milestones': MILESTONES,
                'scheduler_rate': 0.1, 'visual_length': 256,
                'feature_root': str(FEATURE_ROOT),
                'train_list': str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv')}
    if output_model.is_file():
        if not config_path.is_file() or not summary_path.is_file():
            raise RuntimeError(f'{variant}: model có nhưng thiếu config/summary; hãy đổi RUN_TAG.')
        config = json.loads(config_path.read_text(encoding='utf-8'))
        if any(config.get(key) != value for key, value in expected.items()):
            raise RuntimeError(f'{variant}: checkpoint không khớp cấu hình; hãy đổi RUN_TAG.')
        print('Bỏ qua train, model đã có:', output_model)
    else:
        train_cmd = [sys.executable, '-u', str(SRC_DIR / 'ucf_train_cnn_vadclip.py'),
                     '--feature-root', str(FEATURE_ROOT), '--train-list', expected['train_list'],
                     '--projection-width', str(width), '--cnn-channels', str(width),
                     '--kernel-size', '5', '--score-mode', 'scoring_head',
                     '--residual-mode', 'on', '--aggregation', 'topk',
                     '--lambda-alignment', '1.0', '--lambda-text', '1.0',
                     '--seed', str(SEED), '--data-seed', str(DATA_SEED),
                     '--batch-size', '64', '--lr', '1e-5', '--max-epoch', str(MAX_EPOCH),
                     '--min-epochs', str(MIN_EPOCHS),
                     '--early-stopping-patience', str(PATIENCE),
                     '--early-stopping-min-delta', str(MIN_DELTA),
                     '--scheduler-milestones', *(str(m) for m in MILESTONES), '--scheduler-rate', '0.1',
                     '--num-workers', '2', '--visual-length', '256',
                     '--epoch-checkpoint-dir', str(checkpoint_dir),
                     '--output-model-path', str(output_model),
                     '--results-dir', str(train_dir)]
        run_command(train_cmd)
    if not output_model.is_file() or not summary_path.is_file():
        raise FileNotFoundError(f'Missing model or early-stopping summary for {variant}')
    training_summary = json.loads(summary_path.read_text(encoding='utf-8'))
    if training_summary.get('monitor') != 'train_epoch_loss':
        raise RuntimeError(f'{variant}: unexpected early-stopping monitor')
    selected_epoch = training_summary['best_epoch']
    if not (checkpoint_dir / f'epoch_{selected_epoch:03d}.pth').is_file():
        raise FileNotFoundError(f'{variant}: missing selected epoch checkpoint')
    metrics_path = eval_dir / 'metrics.json'
    if metrics_path.is_file():
        old = json.loads(metrics_path.read_text(encoding='utf-8'))
        if old.get('model_path') != str(output_model) or old.get('projection_width') != width or old.get('cnn_channels') != width:
            raise RuntimeError(f'{variant}: metrics không đúng checkpoint/cấu hình; hãy đổi RUN_TAG.')
        print('Bỏ qua evaluate, đã có metrics:', metrics_path)
    else:
        eval_cmd = [sys.executable, '-u', str(SRC_DIR / 'ucf_evaluate_cnn_vadclip.py'),
                    '--model-type', 'cnn', '--model-path', str(output_model),
                    '--feature-root', str(FEATURE_ROOT),
                    '--test-list', str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv'),
                    '--gt-path', str(LIST_DIR / 'gt_ucf.npy'),
                    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
                    '--gt-label-path', str(LIST_DIR / 'gt_label_ucf.npy'),
                    '--projection-width', str(width), '--cnn-channels', str(width),
                    '--kernel-size', '5', '--score-mode', 'scoring_head',
                    '--residual-mode', 'on', '--train-config-json', str(config_path),
                    '--evaluate-dir', str(eval_dir), '--analyze-dir', str(analyze_dir)]
        run_command(eval_cmd)
    results[width] = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('Epoch được chọn theo train loss:', selected_epoch, '/', training_summary['epochs_ran'])
    print('Model được đánh giá:', output_model)
    print('Train:', train_dir, 'Evaluate:', eval_dir, 'Analyze:', analyze_dir)

reference_tag = 'cnn_k5_p128_c64_vs_c128_s234'
reference_variant = 'k5_head_topk_p128_c128'
reference_config = OUTPUT_ROOT / 'train' / reference_tag / reference_variant / 'config.json'
reference_metrics = OUTPUT_ROOT / 'evaluate' / reference_tag / reference_variant / 'metrics.json'
if reference_config.is_file() and reference_metrics.is_file():
    config = json.loads(reference_config.read_text(encoding='utf-8'))
    if (config.get('projection_width'), config.get('cnn_channels'), config.get('data_seed'), config.get('seed')) != (128, 128, DATA_SEED, SEED):
        raise RuntimeError('Kết quả đối chứng P=C=128 không đúng cấu hình/seed.')
    results[128] = json.loads(reference_metrics.read_text(encoding='utf-8'))
    print('Lưu ý: P=C=128 là kết quả 10 epoch với lịch LR cũ, chỉ để tham khảo.')
else:
    print('Không thấy kết quả P=C=128; chỉ so sánh hai lượt mới.')
keys = ('classifier_auc', 'classifier_ap', 'alignment_auc', 'alignment_ap', 'ano_auc', 'avg_mAP')
print('Model | AUC1 | AP1 | AUC2 | AP2 | Ano-AUC | mAP')
for width in (128, 256, 512):
    if width in results:
        print(f'P=C={width} | ' + ' | '.join(f'{results[width][key]:.2f}' for key in keys))
print('Delta 512-256 | ' + ' | '.join(f'{results[512][key]-results[256][key]:+.2f}' for key in keys))
for width in WIDTHS:
    variant = f'k5_head_topk_p{width}_c{width}'
    summary_path = OUTPUT_ROOT / 'analyze' / RUN_TAG / variant / 'summary.json'
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(f'P=C={width} Top-15 anomaly fraction (macro): {summary["selected_anomaly_fraction_abnormal_video_macro"]:.3f}')
